# 05 Model TimeGPT - Rolling Backtest 24h

## Purpose
Run TimeGPT on the same data split and day-ahead `h=24` forecast scope to compare with local models.

## Notes
1. Requires a valid API key (`NIXTLA_API_KEY`).
2. Rolling daily evaluation with 365 windows can be API-intensive.
3. For fast iteration, start with `N_WINDOWS=30` and then scale to full-year.

## Why `cross_validation` here
TimeGPT client supports rolling windows natively via `cross_validation`, which cleanly matches your backtest protocol.

In [ ]:
# If needed:
# %pip install nixtla scikit-learn python-dotenv

import os
import time
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, mean_squared_error
from nixtla import NixtlaClient


In [ ]:
# Config
DATA_PATH = '../../data_cleaned/merged/02_4_clean_data.csv'
H = 24
N_WINDOWS = 30       # start small; increase gradually (e.g., 120, then 365)
STEP_SIZE = 24
CHUNK_WINDOWS = 10   # windows per API call to reduce timeout risk
MAX_RETRIES = 3
RETRY_SLEEP_SEC = 5

start_date = pd.Timestamp('2019-01-01 00:00:00')
split_date = pd.Timestamp('2025-01-01 00:00:00')
end_date = pd.Timestamp('2026-01-01 00:00:00')


In [ ]:
# Load and align schema for TimeGPT
df = pd.read_csv(DATA_PATH)
df['period_start_utc'] = pd.to_datetime(df['period_start_utc'], errors='coerce', utc=True).dt.tz_localize(None)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df[(df['date'] >= start_date) & (df['date'] < end_date)].copy()
df = pd.get_dummies(df, columns=['year'], drop_first=True)

# Keep TimeGPT payload compact to reduce timeout risk
exog_cols = [
    'load_forecast_da', 'res_sum_da', 'gen_forecast_da',
    'dayofyear_sin1', 'dayofyear_cos1',
    'hour_sin1', 'hour_cos1',
    'dayofweek_sin1', 'dayofweek_cos1',
    'is_holiday', 'price_gas'
]

ts = df.rename(columns={'period_start_utc': 'ds', 'price': 'y'})[['ds', 'y'] + exog_cols].copy()
ts['unique_id'] = 'de_price'
ts = ts[['unique_id', 'ds', 'y'] + exog_cols].sort_values(['unique_id', 'ds']).reset_index(drop=True)

print(ts.head())
print(f'Rows: {len(ts):,} | Exogenous: {len(exog_cols)}')


  unique_id                  ds      y  load_forecast_da  res_sum_da  \
0  de_price 2019-01-01 00:00:00  10.07        41692.5650  25669.0050   
1  de_price 2019-01-01 01:00:00  -4.08        40587.2325  27384.1025   
2  de_price 2019-01-01 02:00:00  -9.91        40308.2500  29010.1275   
3  de_price 2019-01-01 03:00:00  -7.41        40659.5750  30359.5675   
4  de_price 2019-01-01 04:00:00 -12.55        40947.7300  31409.2100   

   gen_forecast_da  dayofyear_sin1  dayofyear_cos1  hour_sin1  hour_cos1  \
0         51084.26             0.0             1.0   0.000000   1.000000   
1         51512.67             0.0             1.0   0.258819   0.965926   
2         52693.45             0.0             1.0   0.500000   0.866025   
3         53666.46             0.0             1.0   0.707107   0.707107   
4         54161.85             0.0             1.0   0.866025   0.500000   

   dayofweek_sin1  dayofweek_cos1  is_holiday  price_gas  year_2020  \
0        0.781831         0.62349      

In [10]:
# Init client
# Load key from project .env (works when running notebook from notebook/05_models)
load_dotenv('../../.env')
api_key = os.getenv('NIXTLA_API_KEY')
if not api_key:
    raise ValueError('NIXTLA_API_KEY is not set. Add it to .env or export it in your shell before launching Jupyter.')

# Use longer timeout when supported by installed nixtla version
client_sig = inspect.signature(NixtlaClient)
if 'timeout' in client_sig.parameters:
    client = NixtlaClient(api_key=api_key, timeout=180)
else:
    client = NixtlaClient(api_key=api_key)

print('TimeGPT client initialized')


TimeGPT client initialized


In [11]:
# Rolling daily backtest with chunking + retries to avoid API timeouts
# Strategy: request the latest windows in chunks; then trim tail and repeat.

params = inspect.signature(client.cross_validation).parameters

def run_cv_chunk(df_in: pd.DataFrame, n_windows_chunk: int) -> pd.DataFrame:
    kwargs = {
        'df': df_in,
        'h': H,
        'freq': 'h',
        'id_col': 'unique_id',
        'time_col': 'ds',
        'target_col': 'y',
        'n_windows': n_windows_chunk,
        'step_size': STEP_SIZE,
        'level': None,
    }

    optional = {
        'model': 'timegpt-1',
        'refit': True,
        'hist_exog_list': exog_cols,
        'X_df': df_in[['unique_id', 'ds'] + exog_cols],
    }
    for k, v in optional.items():
        if k in params:
            kwargs[k] = v

    return client.cross_validation(**kwargs)

remaining = N_WINDOWS
offset_rows = 0
cv_parts = []

while remaining > 0:
    chunk_n = min(CHUNK_WINDOWS, remaining)

    if offset_rows > 0:
        df_chunk = ts.iloc[:-offset_rows].copy()
    else:
        df_chunk = ts.copy()

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            part = run_cv_chunk(df_chunk, chunk_n)
            cv_parts.append(part)
            success = True
            print(f'Chunk ok: windows={chunk_n}, remaining_after={remaining - chunk_n}, offset_rows={offset_rows}')
            break
        except Exception as e:
            print(f'Chunk failed (attempt {attempt}/{MAX_RETRIES}): {e}')
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP_SEC)

    if not success:
        raise RuntimeError(f'Failed chunk with n_windows={chunk_n}. Reduce CHUNK_WINDOWS and retry.')

    remaining -= chunk_n
    offset_rows += chunk_n * STEP_SIZE

cv = pd.concat(cv_parts, ignore_index=True)
cv = cv.drop_duplicates(subset=[c for c in ['unique_id', 'cutoff', 'ds'] if c in cv.columns]).reset_index(drop=True)
print(f'CV rows: {len(cv):,}')
cv.head()


INFO:nixtla.nixtla_client:Validating inputs...
INFO:nixtla.nixtla_client:Preprocessing dataframes...
INFO:nixtla.nixtla_client:Querying model metadata...
INFO:nixtla.nixtla_client:Using historical exogenous features: ['load_forecast_da', 'res_sum_da', 'gen_forecast_da', 'dayofyear_sin1', 'dayofyear_cos1', 'hour_sin1', 'hour_cos1', 'dayofweek_sin1', 'dayofweek_cos1', 'is_holiday', 'price_gas', 'year_2020', 'year_2021', 'year_2022', 'year_2023', 'year_2024', 'year_2025']
INFO:nixtla.nixtla_client:Calling Cross Validation Endpoint...
ERROR:nixtla.nixtla_client:Attempt 1 failed with error: The read operation timed out
ERROR:nixtla.nixtla_client:Attempt 2 failed with error: The read operation timed out
ERROR:nixtla.nixtla_client:Attempt 3 failed with error: The read operation timed out
ERROR:nixtla.nixtla_client:Attempt 4 failed with error: The read operation timed out
ERROR:nixtla.nixtla_client:Attempt 5 failed with error: The read operation timed out
ERROR:nixtla.nixtla_client:Attempt 6 f

ReadTimeout: The read operation timed out

In [ ]:
# Standardize prediction column name
pred_col_candidates = [c for c in cv.columns if c.lower().startswith('timegpt')]
if len(pred_col_candidates) == 0:
    raise ValueError(f'Could not find TimeGPT prediction column in: {cv.columns.tolist()}')
pred_col = pred_col_candidates[0]

results = cv.rename(columns={pred_col: 'prediction', 'y': 'actual'})[['unique_id', 'cutoff', 'ds', 'actual', 'prediction']].copy()
results.head()


In [ ]:
# Metrics
rmse = mean_squared_error(results['actual'], results['prediction'], squared=False)
mae = mean_absolute_error(results['actual'], results['prediction'])
eps = 1e-8
mape = np.mean(np.abs((results['actual'] - results['prediction']) / np.maximum(np.abs(results['actual']), eps))) * 100
print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'MAPE: {mape:,.2f}%')


In [ ]:
# Plot
plot_df = results.set_index('ds').sort_index()
ax = plot_df[['actual']].plot(figsize=(15, 5), title='TimeGPT Rolling Daily Backtest (h=24)')
plot_df['prediction'].plot(ax=ax, alpha=0.85)
ax.legend(['actual', 'prediction'])
plt.show()


In [ ]:
# Unified metrics + export for comparison
from pathlib import Path

# normalize evaluation frame name
if 'results' in locals():
    eval_df = results.copy()
elif 'res' in locals():
    eval_df = res.copy()
else:
    raise ValueError('No results dataframe found (expected `results` or `res`).')

# normalize column names
if 'y' in eval_df.columns and 'actual' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'y': 'actual'})
if 'xgb' in eval_df.columns and 'prediction' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'xgb': 'prediction'})
if 'ds' not in eval_df.columns and eval_df.index.name is not None:
    eval_df = eval_df.reset_index()

required_cols = {'actual', 'prediction'}
missing = required_cols - set(eval_df.columns)
if missing:
    raise ValueError(f'Missing required columns for metrics/export: {missing}')

# robust metrics for power prices (can be near zero/negative)
rmse = mean_squared_error(eval_df['actual'], eval_df['prediction'], squared=False)
mae = mean_absolute_error(eval_df['actual'], eval_df['prediction'])
smape = 100 * np.mean(
    2 * np.abs(eval_df['actual'] - eval_df['prediction']) /
    (np.abs(eval_df['actual']) + np.abs(eval_df['prediction']) + 1e-8)
)

mask = np.abs(eval_df['actual']) >= 10
mape_filtered = (
    np.mean(
        np.abs((eval_df.loc[mask, 'actual'] - eval_df.loc[mask, 'prediction']) /
               np.abs(eval_df.loc[mask, 'actual']))
    ) * 100
    if mask.any() else np.nan
)

print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'sMAPE: {smape:,.2f}%')
print(f'MAPE (|actual|>=10): {mape_filtered:,.2f}%')

out_dir = Path('../../artifacts/model_results')
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    'model': 'TimeGPT',
    'rmse': rmse,
    'mae': mae,
    'smape': smape,
    'mape_filtered_abs_ge_10': mape_filtered,
    'n_predictions': len(eval_df)
}])

metrics_df.to_csv(out_dir / 'timegpt_metrics_rolling_24h.csv', index=False)

pred_cols = [c for c in ['unique_id', 'cutoff', 'ds', 'actual', 'prediction'] if c in eval_df.columns]
eval_df[pred_cols].to_csv(out_dir / 'timegpt_predictions_rolling_24h.csv', index=False)

metrics_df
